In [6]:
from google.colab import drive, userdata
import os

drive.mount('/content/drive')
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install trl transformers peft datasets pillow huggingface_hub \
             pyyaml bitsandbytes accelerate scikit-learn -q

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [7]:
from google.colab import userdata
import os

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

!git clone https://github.com/Lifunn/Pipeline-Training-SFT-DFK.git /content/pipeline
%cd /content/pipeline

fatal: destination path '/content/pipeline' already exists and is not an empty directory.
/content/pipeline


In [8]:
from huggingface_hub import hf_hub_download

for filename in ["train_image_text.jsonl", "validation_image_text.jsonl"]:
    hf_hub_download(
        repo_id   = "aitf-its-tim3-dfk/aitf-dfk3-vlm-dataset-jsonl",
        repo_type = "dataset",
        filename  = filename,
        token     = os.environ["HF_TOKEN"],
        local_dir = "/content",
    )
    print(f"✓ {filename}")

✓ train_image_text.jsonl
✓ validation_image_text.jsonl


# updt params (batch size, grad, dll)

In [10]:
import yaml

DRIVE = "/content/drive/MyDrive"

with open("config.yaml") as f:
    cfg = yaml.safe_load(f)

# ── Adjust sesuai GPU ──────────────────────────────────────────────
cfg["training"]["per_device_train_batch_size"] = 2
cfg["training"]["gradient_accumulation_steps"] = 8
cfg["training"]["num_train_epochs"]            = 3
cfg["model"]["max_seq_length"]                 = 2048
# ──────────────────────────────────────────────────────────────────

cfg["data"]["train_file"] = "/content/train_image_text.jsonl"
cfg["data"]["eval_file"]  = "/content/validation_image_text.jsonl"
cfg["data"]["image_dir"]  = f"{DRIVE}/hf_images_folder"

with open("config.yaml", "w") as f:
    yaml.dump(cfg, f)

print("Config updated ✓")
print(f"  train      : {cfg['data']['train_file']}")
print(f"  eval       : {cfg['data']['eval_file']}")
print(f"  image_dir  : {cfg['data']['image_dir']}")
print(f"  batch_size : {cfg['training']['per_device_train_batch_size']}")
print(f"  grad_accum : {cfg['training']['gradient_accumulation_steps']}")
print(f"  effective BS: {cfg['training']['per_device_train_batch_size'] * cfg['training']['gradient_accumulation_steps']}")

Config updated ✓
  train      : /content/train_image_text.jsonl
  eval       : /content/validation_image_text.jsonl
  image_dir  : /content/drive/MyDrive/hf_images_folder
  batch_size : 2
  grad_accum : 8
  effective BS: 16


In [11]:
!git pull origin main

From https://github.com/Lifunn/Pipeline-Training-SFT-DFK
 * branch            main       -> FETCH_HEAD
Already up to date.


# Smoke TEST


In [13]:
import unsloth
!python train.py --smoke_test --hf_token $HF_TOKEN

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
/usr/local/lib/python3.12/dist-packages/huggingface_hub/constants.py:294: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
19:56:47 [INFO] train

In [14]:
import shutil

src = "/content/pipeline"
dst = "/content/drive/MyDrive/dfk_sft_output-updt"

shutil.copytree(src, dst, dirs_exist_ok=True)
print(f"Model tersimpan ke Drive: {dst} ✓")

Model tersimpan ke Drive: /content/drive/MyDrive/dfk_sft_output-updt ✓


# Full Training

In [15]:
# !python train.py --hf_token $HF_TOKEN

# Evaluasi

In [16]:
# !python evaluate.py \
#     --model_dir ./output/dfk_sft \
#     --max_samples 300 \
#     --output_json ./output/dfk_sft/eval_results.json \
#     --hf_token $HF_TOKEN